# Multiscale Cantor geometry × neural scaling

This corrected experiment controls:

\[
d_q=q\frac{\log 2}{\log 3}
\]

using \(C^q\), and teacher regularity \(\beta\) using a hierarchical random function

\[
f^*(x)=\sum_{k=1}^{K}3^{-\beta k}\xi_{k,\mathrm{cell}_k(x)}.
\]

We test

\[
\alpha_N \stackrel{?}{\propto}\frac{\beta}{d},
\qquad
\alpha_D \stackrel{?}{\propto}\frac{\beta}{d}.
\]

**Critical correction for the \(D\)-experiment:** every dataset size receives exactly the same number of optimizer updates, with minibatches sampled with replacement.

Run `FAST` first. Do not run `FULL` until we inspect FAST.

In [ ]:
!pip -q install scipy pandas matplotlib

In [ ]:
import gc, math, os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import least_squares
from scipy.stats import linregress, spearmanr

warnings.filterwarnings("ignore")
SEED = 20260813

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
RUN_MODE = "FAST"

AMBIENT_DIM = 8
CANTOR_DEPTH = 14

if RUN_MODE == "FAST":
    Q_VALUES = [1, 2, 4, 6, 8]
    BETA_VALUES = [0.5, 1.0]
    WIDTHS_N = [16, 32, 64, 128, 256]
    D_VALUES = [512, 2_000, 8_000, 32_000]
    REPEATS = 2

    N_TRAIN_STEPS = 2500
    N_BATCH = 1024

    D_WIDTH = 256
    D_TRAIN_STEPS = 3500
    D_BATCH = 512

    TEST_SIZE = 12_000
else:
    Q_VALUES = list(range(1, 9))
    BETA_VALUES = [0.35, 0.5, 0.75, 1.0, 1.25]
    WIDTHS_N = [16, 24, 32, 48, 64, 96, 128, 192, 256, 384]
    D_VALUES = [512, 1_000, 2_000, 4_000, 8_000, 16_000, 32_000, 64_000, 128_000]
    REPEATS = 4

    N_TRAIN_STEPS = 6000
    N_BATCH = 2048

    D_WIDTH = 384
    D_TRAIN_STEPS = 7000
    D_BATCH = 512

    TEST_SIZE = 30_000

SAVE_DIR = "/content"
os.makedirs(SAVE_DIR, exist_ok=True)

print("q:", Q_VALUES)
print("beta:", BETA_VALUES)
print("N widths:", WIDTHS_N)
print("D:", D_VALUES)
print("repeats:", REPEATS)
print("equal D optimizer steps:", D_TRAIN_STEPS)

In [ ]:
LOG2_LOG3 = math.log(2.0) / math.log(3.0)

display(pd.DataFrame({
    "q": Q_VALUES,
    "dimension": [q * LOG2_LOG3 for q in Q_VALUES]
}))

## Cantor support and addresses

We generate the binary Cantor address explicitly, then construct the Euclidean coordinates and randomly rotate them inside the common ambient space \(\mathbb R^8\).

In [ ]:
def random_orthogonal(dim, seed):
    g = torch.Generator(device="cpu")
    g.manual_seed(int(seed))
    A = torch.randn(dim, dim, generator=g, dtype=torch.float32)
    Q, R = torch.linalg.qr(A)
    signs = torch.sign(torch.diag(R))
    signs[signs == 0] = 1
    return (Q * signs).float()

class CantorSupport:
    def __init__(self, q, ambient_dim=8, depth=14, rotation_seed=0):
        self.q = int(q)
        self.ambient_dim = int(ambient_dim)
        self.depth = int(depth)
        self.dimension = self.q * LOG2_LOG3
        self.rotation = random_orthogonal(self.ambient_dim, rotation_seed)
        self.powers = 3.0 ** torch.arange(1, self.depth + 1, dtype=torch.float32)

    def sample_with_address(self, n, seed, device="cpu"):
        g = torch.Generator(device="cpu")
        g.manual_seed(int(seed))

        bits = torch.randint(
            0, 2, (n, self.q, self.depth),
            generator=g, dtype=torch.int64
        )

        coords = (
            2.0 * (bits.float() / self.powers.view(1,1,-1)).sum(dim=-1)
            - 1.0
        )

        X = torch.zeros(n, self.ambient_dim, dtype=torch.float32)
        X[:, :self.q] = coords
        X = X @ self.rotation.T

        return X.to(device), bits.to(device)

## Hierarchical multiscale teacher

At Cantor level \(k\), each point receives a deterministic pseudo-random coefficient depending on its prefix cell. The level-\(k\) amplitude is \(3^{-\beta k}\).

In [ ]:
def pseudo_cell_value(bits, level, seed):
    # bits: [n,q,depth]
    n, q, depth = bits.shape

    # int64 hash state; overflow is intentional
    h = torch.full(
        (n,),
        int(seed) % 2_000_000_000,
        dtype=torch.int64,
        device=bits.device
    )

    # FNV-like integer mixing, kept inside int64-safe operations
    prime = 1_000_003

    for j in range(q):
        for k in range(level):
            token = bits[:, j, k].to(torch.int64) + 3*(j+1) + 131*(k+1)
            h = (h * prime + token + 97) % 2_147_483_647

    # deterministic nonlinear mixing
    h = (1103515245 * h + 12345) % 2_147_483_647

    # map to approximately [-1,1]
    return 2.0 * h.float() / 2_147_483_646.0 - 1.0

class HierarchicalCantorTeacher:
    def __init__(self, beta, depth=14, seed=0):
        self.beta = float(beta)
        self.depth = int(depth)
        self.seed = int(seed)
        self.weights = torch.tensor([
            3.0 ** (-self.beta * k)
            for k in range(1, self.depth + 1)
        ], dtype=torch.float32)

    @torch.no_grad()
    def __call__(self, bits):
        y = torch.zeros(bits.shape[0], device=bits.device, dtype=torch.float32)
        w = self.weights.to(bits.device)

        for level in range(1, self.depth + 1):
            xi = pseudo_cell_value(
                bits,
                level,
                self.seed + 10_000 * level
            )
            y = y + w[level-1] * xi

        return y

In [ ]:
@torch.no_grad()
def estimate_normalization(support, teacher, seed, n=20_000):
    _, bits = support.sample_with_address(n, seed=seed, device="cpu")
    y = teacher(bits)
    return float(y.mean()), float(y.std().clamp_min(1e-6))

@torch.no_grad()
def sample_labeled(support, teacher, mu, sd, n, seed, device="cpu"):
    X, bits = support.sample_with_address(n, seed=seed, device=device)
    y = (teacher(bits) - mu) / sd
    return X, y

## Student network

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=8, width=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

for w in WIDTHS_N:
    m = MLP(AMBIENT_DIM, w)
    print(f"width={w:4d} params={count_params(m):8d}")

In [ ]:
@torch.inference_mode()
def evaluate_mse(model, X, y, batch_size=4096):
    model.eval()
    total = 0.0
    n = len(X)

    for i in range(0, n, batch_size):
        xb = X[i:i+batch_size].to(DEVICE)
        yb = y[i:i+batch_size].to(DEVICE)
        pred = model(xb)
        total += F.mse_loss(pred, yb, reduction="sum").item()

    return total / n

def fit_power_law_with_floor(x, loss):
    x = np.asarray(x, float)
    y = np.asarray(loss, float)
    order = np.argsort(x)
    x, y = x[order], y[order]

    ymin = y.min()
    spread = max(float(np.ptp(y)), max(ymin, 1e-8))

    theta0 = np.array([max(0.0, 0.5*ymin), np.log(spread), 0.5])
    lo = np.array([0.0, -40.0, 1e-4])
    hi = np.array([max(ymin*0.999, 1e-12), 40.0, 6.0])

    def pred(th):
        E, logA, alpha = th
        return E + np.exp(logA) * x**(-alpha)

    res = least_squares(
        lambda th: pred(th) - y,
        theta0,
        bounds=(lo, hi),
        max_nfev=100_000
    )

    E, logA, alpha = res.x
    yhat = pred(res.x)
    ss_res = ((y-yhat)**2).sum()
    ss_tot = ((y-y.mean())**2).sum()

    return {
        "E": float(E),
        "A": float(np.exp(logA)),
        "alpha": float(alpha),
        "r2": float(1.0 - ss_res/max(ss_tot,1e-15))
    }

def fit_raw_loglog(x, loss):
    lr = linregress(np.log(np.asarray(x,float)), np.log(np.asarray(loss,float)))
    return {
        "alpha_raw": float(-lr.slope),
        "r2_raw": float(lr.rvalue**2)
    }

# Experiment A: online \(N\)-scaling

In [ ]:
def train_online_N(support, teacher, mu, sd, width, seed):
    seed_everything(seed)

    model = MLP(AMBIENT_DIM, width).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-6)

    for step in range(N_TRAIN_STEPS):
        X, y = sample_labeled(
            support, teacher, mu, sd,
            n=N_BATCH,
            seed=seed + 100_000 + step,
            device=DEVICE
        )

        pred = model(X)
        loss = F.mse_loss(pred, y)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

    return model

N_RAW_CSV = os.path.join(SAVE_DIR, "multiscale_N_raw.csv")
n_raw = pd.read_csv(N_RAW_CSV) if os.path.exists(N_RAW_CSV) else pd.DataFrame()

done_N = set()
if len(n_raw):
    done_N = set(zip(
        n_raw["repeat"].astype(int),
        n_raw["q"].astype(int),
        n_raw["beta"].astype(float),
        n_raw["width"].astype(int)
    ))

for rep in range(REPEATS):
    print(f"\n===== N repeat {rep+1}/{REPEATS} =====")

    for q in Q_VALUES:
        support = CantorSupport(
            q=q,
            ambient_dim=AMBIENT_DIM,
            depth=CANTOR_DEPTH,
            rotation_seed=SEED + 1_000_000*rep + 10_000*q
        )

        for beta in BETA_VALUES:
            teacher = HierarchicalCantorTeacher(
                beta=beta,
                depth=CANTOR_DEPTH,
                seed=SEED + 2_000_000*rep + 20_000*q + int(1000*beta)
            )

            mu, sd = estimate_normalization(
                support, teacher,
                seed=SEED + 3_000_000*rep + 30_000*q + int(1000*beta)
            )

            Xtest, ytest = sample_labeled(
                support, teacher, mu, sd,
                n=TEST_SIZE,
                seed=SEED + 4_000_000*rep + 40_000*q + int(1000*beta),
                device="cpu"
            )

            for width in WIDTHS_N:
                key = (rep, q, float(beta), width)
                if key in done_N:
                    print("skip", key)
                    continue

                train_seed = (
                    SEED + 5_000_000*rep + 50_000*q
                    + 1000*width + int(100*beta)
                )

                model = train_online_N(
                    support, teacher, mu, sd, width, train_seed
                )

                mse = evaluate_mse(model, Xtest, ytest)

                row = pd.DataFrame([{
                    "repeat": rep,
                    "q": q,
                    "dimension": support.dimension,
                    "beta": beta,
                    "beta_over_d": beta/support.dimension,
                    "width": width,
                    "params": count_params(model),
                    "test_mse": mse
                }])

                n_raw = pd.concat([n_raw,row], ignore_index=True)
                n_raw.to_csv(N_RAW_CSV,index=False)

                print(
                    f"q={q}, d={support.dimension:.3f}, beta={beta:.2f}, "
                    f"N={count_params(model)}, mse={mse:.6g}"
                )

                del model
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

print("saved:", N_RAW_CSV)

In [ ]:
n_mean = (
    n_raw
    .groupby(["q","dimension","beta","beta_over_d","params"], as_index=False)
    .agg(
        mean_mse=("test_mse","mean"),
        sd_mse=("test_mse","std"),
        n=("test_mse","size")
    )
)

rows=[]
for keys,g in n_mean.groupby(["q","dimension","beta","beta_over_d"]):
    q,d,beta,bod = keys
    g=g.sort_values("params")
    rows.append({
        "q":q,"dimension":d,"beta":beta,"beta_over_d":bod,
        **fit_power_law_with_floor(g.params,g.mean_mse),
        **fit_raw_loglog(g.params,g.mean_mse)
    })

n_exp=pd.DataFrame(rows).sort_values(["beta","dimension"])
display(n_exp)

N_EXP_CSV=os.path.join(SAVE_DIR,"multiscale_N_exponents.csv")
n_exp.to_csv(N_EXP_CSV,index=False)

In [ ]:
for beta in BETA_VALUES:
    plt.figure(figsize=(7,5))
    z=n_mean[n_mean.beta==beta]
    for q,g in z.groupby("q"):
        g=g.sort_values("params")
        d=float(g.dimension.iloc[0])
        plt.plot(g.params,g.mean_mse,marker="o",label=f"q={q}, d={d:.2f}")
    plt.xscale("log"); plt.yscale("log")
    plt.xlabel("parameter count N"); plt.ylabel("test MSE")
    plt.title(f"N-scaling, beta={beta}")
    plt.legend(fontsize=8)
    plt.show()

# Experiment B: \(D\)-scaling with equal training steps

In [ ]:
def train_fixed_dataset_equal_steps(Xtrain, ytrain, width, seed):
    seed_everything(seed)

    model = MLP(AMBIENT_DIM, width).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-6)

    Xtrain = Xtrain.to(DEVICE)
    ytrain = ytrain.to(DEVICE)
    n = len(Xtrain)

    g = torch.Generator(device=DEVICE)
    g.manual_seed(seed + 12345)

    for step in range(D_TRAIN_STEPS):
        idx = torch.randint(
            0, n, (D_BATCH,),
            generator=g,
            device=DEVICE
        )

        pred = model(Xtrain[idx])
        loss = F.mse_loss(pred, ytrain[idx])

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

    return model

D_RAW_CSV = os.path.join(SAVE_DIR, "multiscale_D_raw.csv")
d_raw = pd.read_csv(D_RAW_CSV) if os.path.exists(D_RAW_CSV) else pd.DataFrame()

done_D=set()
if len(d_raw):
    done_D=set(zip(
        d_raw["repeat"].astype(int),
        d_raw["q"].astype(int),
        d_raw["beta"].astype(float),
        d_raw["D"].astype(int)
    ))

for rep in range(REPEATS):
    print(f"\n===== D repeat {rep+1}/{REPEATS} =====")

    for q in Q_VALUES:
        support=CantorSupport(
            q=q,
            ambient_dim=AMBIENT_DIM,
            depth=CANTOR_DEPTH,
            rotation_seed=SEED + 1_000_000*rep + 10_000*q
        )

        for beta in BETA_VALUES:
            teacher=HierarchicalCantorTeacher(
                beta=beta,
                depth=CANTOR_DEPTH,
                seed=SEED + 2_000_000*rep + 20_000*q + int(1000*beta)
            )

            mu,sd=estimate_normalization(
                support,teacher,
                seed=SEED + 3_000_000*rep + 30_000*q + int(1000*beta)
            )

            Xtest,ytest=sample_labeled(
                support,teacher,mu,sd,
                n=TEST_SIZE,
                seed=SEED + 9_000_000*rep + 90_000*q + int(1000*beta),
                device="cpu"
            )

            for D in D_VALUES:
                key=(rep,q,float(beta),int(D))
                if key in done_D:
                    print("skip",key)
                    continue

                Xtrain,ytrain=sample_labeled(
                    support,teacher,mu,sd,
                    n=int(D),
                    seed=SEED + 6_000_000*rep + 60_000*q + int(D) + int(1000*beta),
                    device="cpu"
                )

                model=train_fixed_dataset_equal_steps(
                    Xtrain,ytrain,D_WIDTH,
                    seed=SEED + 7_000_000*rep + 70_000*q + int(D) + int(1000*beta)
                )

                train_mse=evaluate_mse(model,Xtrain,ytrain)
                test_mse=evaluate_mse(model,Xtest,ytest)

                row=pd.DataFrame([{
                    "repeat":rep,
                    "q":q,
                    "dimension":support.dimension,
                    "beta":beta,
                    "beta_over_d":beta/support.dimension,
                    "D":int(D),
                    "width":D_WIDTH,
                    "params":count_params(model),
                    "train_steps":D_TRAIN_STEPS,
                    "train_mse":train_mse,
                    "test_mse":test_mse
                }])

                d_raw=pd.concat([d_raw,row],ignore_index=True)
                d_raw.to_csv(D_RAW_CSV,index=False)

                print(
                    f"q={q}, d={support.dimension:.3f}, beta={beta:.2f}, "
                    f"D={D}, train={train_mse:.5g}, test={test_mse:.5g}"
                )

                del model
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

print("saved:",D_RAW_CSV)

In [ ]:
d_mean=(
    d_raw
    .groupby(["q","dimension","beta","beta_over_d","D"],as_index=False)
    .agg(
        mean_test_mse=("test_mse","mean"),
        sd_test_mse=("test_mse","std"),
        mean_train_mse=("train_mse","mean"),
        n=("test_mse","size")
    )
)

rows=[]
for keys,g in d_mean.groupby(["q","dimension","beta","beta_over_d"]):
    q,d,beta,bod=keys
    g=g.sort_values("D")
    rows.append({
        "q":q,"dimension":d,"beta":beta,"beta_over_d":bod,
        **fit_power_law_with_floor(g.D,g.mean_test_mse),
        **fit_raw_loglog(g.D,g.mean_test_mse)
    })

d_exp=pd.DataFrame(rows).sort_values(["beta","dimension"])
display(d_exp)

D_EXP_CSV=os.path.join(SAVE_DIR,"multiscale_D_exponents.csv")
d_exp.to_csv(D_EXP_CSV,index=False)

In [ ]:
for beta in BETA_VALUES:
    plt.figure(figsize=(7,5))
    z=d_mean[d_mean.beta==beta]
    for q,g in z.groupby("q"):
        g=g.sort_values("D")
        d=float(g.dimension.iloc[0])
        plt.plot(g.D,g.mean_test_mse,marker="o",label=f"q={q}, d={d:.2f}")
    plt.xscale("log"); plt.yscale("log")
    plt.xlabel("dataset size D"); plt.ylabel("test MSE")
    plt.title(f"D-scaling, beta={beta}")
    plt.legend(fontsize=8)
    plt.show()

# Main tests: exponent versus \(\beta/d\)

In [ ]:
def through_origin_fit(x,y):
    x=np.asarray(x,float)
    y=np.asarray(y,float)
    c=float((x@y)/(x@x))
    pred=c*x
    mse=float(np.mean((y-pred)**2))
    return c,pred,mse

# N
xN=n_exp.beta_over_d.to_numpy()
yN=n_exp.alpha.to_numpy()
cN,predN,mseN=through_origin_fit(xN,yN)
rhoN,pN=spearmanr(xN,yN)

# D
xD=d_exp.beta_over_d.to_numpy()
yD=d_exp.alpha.to_numpy()
cD,predD,mseD=through_origin_fit(xD,yD)
rhoD,pD=spearmanr(xD,yD)

print("N: Spearman=",rhoN,"p=",pN,"c=",cN,"MSE=",mseN)
print("D: Spearman=",rhoD,"p=",pD,"c=",cD,"MSE=",mseD)

plt.figure(figsize=(6,5))
plt.scatter(xN,yN,s=60)
xx=np.linspace(xN.min(),xN.max(),200)
plt.plot(xx,cN*xx)
plt.xlabel("beta / d"); plt.ylabel("alpha_N")
plt.title(f"N: alpha vs beta/d, c≈{cN:.3f}")
plt.show()

plt.figure(figsize=(6,5))
plt.scatter(xD,yD,s=60)
xx=np.linspace(xD.min(),xD.max(),200)
plt.plot(xx,cD*xx)
plt.xlabel("beta / d"); plt.ylabel("alpha_D")
plt.title(f"D: alpha vs beta/d, c≈{cD:.3f}")
plt.show()

# Compare two candidate laws

We compare

\[
\alpha=c\frac{\beta}{d}
\]

against a saturating statistical form

\[
\alpha=c\frac{\beta}{2\beta+d}.
\]

In [ ]:
comparison=[]

for name,df in [("N",n_exp),("D",d_exp)]:
    y=df.alpha.to_numpy()

    x1=(df.beta/df.dimension).to_numpy()
    c1,p1,m1=through_origin_fit(x1,y)

    x2=(df.beta/(2*df.beta+df.dimension)).to_numpy()
    c2,p2,m2=through_origin_fit(x2,y)

    comparison.extend([
        {"experiment":name,"law":"c*beta/d","c":c1,"mse":m1},
        {"experiment":name,"law":"c*beta/(2beta+d)","c":c2,"mse":m2}
    ])

law_comparison=pd.DataFrame(comparison)
display(law_comparison)

# Leave-one-\((d,\beta)\)-pair-out prediction

In [ ]:
def loo_inverse(df):
    out=[]
    for idx in df.index:
        train=df.drop(index=idx)
        test=df.loc[idx]

        xtr=(train.beta/train.dimension).to_numpy()
        ytr=train.alpha.to_numpy()
        c=float((xtr@ytr)/(xtr@xtr))

        xte=float(test.beta/test.dimension)
        pred=c*xte
        obs=float(test.alpha)

        out.append({
            "q":int(test.q),
            "dimension":float(test.dimension),
            "beta":float(test.beta),
            "beta_over_d":xte,
            "observed_alpha":obs,
            "predicted_alpha":pred,
            "abs_error":abs(obs-pred),
            "relative_error":abs(obs-pred)/max(abs(obs),1e-12),
            "c_fit_without_point":c
        })

    return pd.DataFrame(out)

loo_N=loo_inverse(n_exp)
loo_D=loo_inverse(d_exp)

print("N mean relative LOO error:",loo_N.relative_error.mean())
print("D mean relative LOO error:",loo_D.relative_error.mean())

display(loo_N)
display(loo_D)

# Optimization diagnostic for the \(D\)-experiment

If small \(D\) has low training error but substantially higher test error, the experiment is genuinely data/generalization limited rather than simply under-optimized.

In [ ]:
for beta in BETA_VALUES:
    plt.figure(figsize=(7,5))
    z=d_mean[d_mean.beta==beta]
    for q,g in z.groupby("q"):
        g=g.sort_values("D")
        plt.plot(g.D,g.mean_train_mse,marker="o",label=f"q={q}")
    plt.xscale("log"); plt.yscale("log")
    plt.xlabel("dataset size D"); plt.ylabel("training MSE")
    plt.title(f"Optimization diagnostic, beta={beta}")
    plt.legend()
    plt.show()

In [ ]:
N_LOO_CSV=os.path.join(SAVE_DIR,"multiscale_N_LOO.csv")
D_LOO_CSV=os.path.join(SAVE_DIR,"multiscale_D_LOO.csv")
COMPARE_CSV=os.path.join(SAVE_DIR,"multiscale_law_comparison.csv")
SUMMARY_CSV=os.path.join(SAVE_DIR,"multiscale_summary.csv")

loo_N.to_csv(N_LOO_CSV,index=False)
loo_D.to_csv(D_LOO_CSV,index=False)
law_comparison.to_csv(COMPARE_CSV,index=False)

summary=pd.DataFrame([
    {
        "experiment":"N",
        "spearman_alpha_vs_beta_over_d":rhoN,
        "p_value":pN,
        "through_origin_c":cN,
        "mean_LOO_relative_error":loo_N.relative_error.mean()
    },
    {
        "experiment":"D",
        "spearman_alpha_vs_beta_over_d":rhoD,
        "p_value":pD,
        "through_origin_c":cD,
        "mean_LOO_relative_error":loo_D.relative_error.mean()
    }
])

summary.to_csv(SUMMARY_CSV,index=False)
display(summary)

In [ ]:
import zipfile

ZIP_PATH=os.path.join(SAVE_DIR,"cantor_multiscale_scaling_results.zip")

paths=[
    N_RAW_CSV,N_EXP_CSV,N_LOO_CSV,
    D_RAW_CSV,D_EXP_CSV,D_LOO_CSV,
    COMPARE_CSV,SUMMARY_CSV
]

with zipfile.ZipFile(ZIP_PATH,"w") as z:
    for path in paths:
        if os.path.exists(path):
            z.write(path,arcname=os.path.basename(path))

print("created:",ZIP_PATH)

try:
    from google.colab import files
    files.download(ZIP_PATH)
except Exception:
    print("download manually:",ZIP_PATH)